In [ ]:
# Step 1: Install xgboost — a powerful prediction algorithm will compare later
# Step 2: Install shap — a tool that explains why the model made each prediction
# The -q flag means "quiet mode" so it does not print lots of installation text
!pip install xgboost shap -q
print("Libraries installed")

In [ ]:
# Step 1: Import the Google Drive connector from Colab
# Step 2: Mount Drive at /content/drive so that can read and write files
# Think of this like plugging in a USB stick before you can use it
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted")

In [ ]:
# Step 1: Import pandas — used to work with tables of data (like Excel in Python)
import pandas as pd

# Step 2: Import numpy — used for maths and number calculations
import numpy as np

# Step 3: Import os and glob — used to find and work with files on disk
import os
import glob

# Step 4: Import matplotlib and seaborn — used to draw charts and graphs
import matplotlib.pyplot as plt
import seaborn as sns

# Step 5: Turn off warning messages so the output stays clean and readable
import warnings
warnings.filterwarnings('ignore')

# Step 6: Set up the folder paths where data lives on Google Drive
# BASE_PATH = the main project folder
# RAW_PATH  = where the original CSV files are stored
# SAVE_PATH = where will save the cleaned data after processing
BASE_PATH = '/content/drive/MyDrive/uwe_project/'
RAW_PATH  = BASE_PATH + 'data/raw/'
SAVE_PATH = BASE_PATH + 'data/processed/'

print("Imports done")
print(f"Looking for data in: {RAW_PATH}")

In [ ]:
# Step 1: Define a function that loads all 13 CUG CSV files (2015 to 2027)
# and stacks them into one big table
def load_all_years(raw_path):

    # Step 2: Find all files in the raw folder that match the pattern cug_*.csv
    all_files = sorted(glob.glob(raw_path + 'cug_*.csv'))

    # Step 3: If no files are found, print a message and stop
    if len(all_files) == 0:
        print(f"No files found in {raw_path}")
        return None

    print(f"Found {len(all_files)} files")

    # Step 4: Create an empty list to store each year's data
    all_years = []

    # Step 5: Loop through each file one by one
    for filepath in all_files:

        # Step 6: Extract the year number from the filename
        # e.g. 'cug_2020.csv' becomes the number 2020
        filename = os.path.basename(filepath)
        year = int(filename.replace('cug_', '').replace('.csv', ''))

        try:
            # Step 7: Read the CSV file
            # skiprows=4 skips the header rows at the top of each CUG file
            # encoding='utf-8-sig' handles an invisible character in the 2015 file
            # on_bad_lines='skip' ignores broken rows instead of crashing
            df = pd.read_csv(
                filepath,
                skiprows=4,
                encoding='utf-8-sig',
                on_bad_lines='skip',
            )

            # Step 8: Add a column to record which year this data came from
            df['year'] = year
            all_years.append(df)
            print(f"  Loaded {year}: {len(df)} universities")

        except Exception as e:
            print(f"  ERROR loading {year}: {e}")

    # Step 9: Stack all 13 years into one single combined table
    combined = pd.concat(all_years, ignore_index=True)
    print(f"\nTotal rows: {len(combined)}")
    print(f"Years: {sorted(combined['year'].unique())}")
    return combined

# Step 10: Run the function and store the result in df_raw
df_raw = load_all_years(RAW_PATH)

In [ ]:
# Step 1: Define a function to standardise column names across all years
# The CUG changed column names over the years, for example:
#'Degree Completion' in older files = 'Continuation' in newer files
#rename everything to one consistent set of names
def standardise_columns(df):

    # Step 2: Create a dictionary that maps old column names to new standard names
    rename_map = {
        'Rank'                          : 'rank',
        'Institution'                   : 'university',
        'Entry Standards'               : 'entry_tariff',
        'Student Satisfaction'          : 'satisfaction',
        'Research Quality'              : 'research_quality',
        'Research Intensity'            : 'research_intensity',
        'Student-Staff Ratio'           : 'staff_ratio',
        'Academic Services Spend'       : 'academic_spend',
        'Facilities Spend'              : 'facilities_spend',
        'Overall Score'                 : 'overall_score',
        'Graduate prospects outcomes'   : 'career_prospects',
        'Graduate Prospects'            : 'career_prospects',
        'Graduate prospects on track'   : 'career_on_track',
        'Continuation'                  : 'continuation',
        'Degree Completion'             : 'continuation',
        'Good Honours'                  : 'good_honours',
    }

    # Step 3: Only rename columns that actually exist in the current file
    # (some columns only appear in certain years)
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)

    # Step 4: Drop columns that only existed in 2015 and would cause problems
    #'Green Score' was discontinued after 2022
    cols_to_drop_2015 = ['Green Score', 'Graduate Prospects']
    cols_to_drop_2015 = [c for c in cols_to_drop_2015 if c in df.columns]
    if cols_to_drop_2015:
        df = df.drop(columns=cols_to_drop_2015)
        print(f"Dropped 2015-specific columns: {cols_to_drop_2015}")

    # Step 5: Remove any duplicate column names that might have appeared
    df = df.loc[:, ~df.columns.duplicated()]

    print("Columns after rename:", list(df.columns))
    return df

# Step 6: Run the function on raw data
df_named = standardise_columns(df_raw)

print("\nYear counts:")
print(df_named['year'].value_counts().sort_index())

In [ ]:
# Step 1: Define a function that fixes common data quality problems
def clean_data(df):

    print(f"Before cleaning: {len(df)} rows")

    #Step 2: Convert text columns to numbers
    #Some columns store numbers as text (e.g. "3.5" as a string)
    #errors='coerce' means: if a value cannot be converted, replace with NaN
    numeric_cols = [
        'rank', 'entry_tariff', 'satisfaction', 'research_quality',
        'research_intensity', 'career_prospects', 'career_on_track',
        'staff_ratio', 'academic_spend', 'facilities_spend',
        'continuation', 'overall_score'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 3: Remove impossible values that are clearly data entry errors
    #Satisfaction can only be between 1 and 4 in the CUG scale
    #Staff ratio above 60 or below 1 is physically impossible
    df.loc[df['satisfaction'] < 1,     'satisfaction']    = np.nan
    df.loc[df['satisfaction'] > 4,     'satisfaction']    = np.nan
    df.loc[df['staff_ratio'] < 1,      'staff_ratio']     = np.nan
    df.loc[df['staff_ratio'] > 60,     'staff_ratio']     = np.nan
    df.loc[df['career_prospects'] < 0,  'career_prospects'] = np.nan
    df.loc[df['career_prospects'] > 100,'career_prospects'] = np.nan
    df.loc[df['continuation'] < 50,    'continuation']    = np.nan
    df.loc[df['continuation'] > 100,   'continuation']    = np.nan

    # Step 4: Standardise all the different name spellings for UWE Bristol
    #The CUG used 5 different names for UWE across different years
    #map them all to 'UWE Bristol' so the model treats them as one institution
    name_fixes = {
        'Bristol, UWE'                              : 'UWE Bristol',
        'UWE, Bristol'                              : 'UWE Bristol',
        'University of the West of England'         : 'UWE Bristol',
        'Bristol, University of the West of England': 'UWE Bristol',
        'West of England, Bristol'                  : 'UWE Bristol',
        'UWE'                                       : 'UWE Bristol',
    }
    df['university'] = df['university'].replace(name_fixes)
    df['university'] = df['university'].str.strip()

    # Step 5: Remove duplicate rows where the same university appears twice in the same year
    before = len(df)
    df = df.drop_duplicates(subset=['university', 'year'])
    print(f"Removed {before - len(df)} duplicate rows")

    # Step 6: Remove rows with no rank it cannot use these for training
    df = df.dropna(subset=['rank'])

    #Step 7: Fill in remaining missing values
    #First i try filling each university's missing values with its own median
    #Then it fill any remaining gaps with the overall column median
    fill_cols = [
        'satisfaction', 'staff_ratio', 'career_prospects',
        'continuation', 'entry_tariff', 'research_quality'
    ]
    for col in fill_cols:
        if col in df.columns:
            df[col] = df.groupby('university')[col].transform(
                lambda x: x.fillna(x.median())
            )
            df[col] = df[col].fillna(df[col].median())

    print(f"After cleaning: {len(df)} rows")
    print("\nMissing values per column:")
    print(df.isnull().sum())
    return df

# Step 8: Run the cleaning function on the renamed data
df_clean = clean_data(df_named)

In [ ]:
#Step 1: Define a function that creates new features from existing columns
#Raw yearly metrics do not capture trends — a university with
#satisfaction = 3.5 that is FALLING is very different from one that is RISING
#create lag features, change features, and rolling averages to capture this
def engineer_features(df):

    # Step 2: Drop columns with too many missing values — they would hurt the model
    cols_to_drop = ['good_honours', 'career_on_track']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped incomplete columns: {cols_to_drop}")

    # Step 3: Sort the data by university name then year
    # This is essential so that .shift(1) looks at the correct previous year
    df = df.sort_values(['university', 'year']).reset_index(drop=True)

    # Step 4: Create LAG features — what was the value LAST year?
    #.shift(1) copies the value from the row above within each university group
    g = df.groupby('university')
    df['rank_lag1']         = g['rank'].shift(1)
    df['satisfaction_lag1'] = g['satisfaction'].shift(1)
    df['staff_ratio_lag1']  = g['staff_ratio'].shift(1)
    df['career_lag1']       = g['career_prospects'].shift(1)

    # Step 5: Create CHANGE features — how much did each metric change from last year?
    # Positive rank_change means the rank got WORSE (higher number = lower position)
    df['rank_change']         = df['rank'] - df['rank_lag1']
    df['satisfaction_change'] = df['satisfaction'] - df['satisfaction_lag1']
    df['staff_ratio_change']  = df['staff_ratio'] - df['staff_ratio_lag1']
    df['career_change']       = df['career_prospects'] - df['career_lag1']

    # Step 6: Create 3-YEAR ROLLING AVERAGES — what has the trend been over 3 years?
    # This smooths out one-off spikes and captures the longer trend direction
    # min_periods=2 means: calculate even if only have 2 years of data
    df['rank_3yr_avg'] = g['rank'].transform(
        lambda x: x.rolling(window=3, min_periods=2).mean()
    )
    df['satisfaction_3yr_avg'] = g['satisfaction'].transform(
        lambda x: x.rolling(window=3, min_periods=2).mean()
    )

    # Step 7: Assign a VOLATILITY LABEL based on how much the rank changed
    # This becomes the target for early warning classifier
    # Positive rank_change = getting WORSE in the table
    def assign_label(change):
        if pd.isna(change):   return None
        elif change >= 20:    return 'collapsing'   # dropped 20+ places
        elif change >= 8:     return 'falling'      # dropped 8-19 places
        elif change > -8:     return 'stable'       # stayed roughly the same
        elif change > -20:    return 'rising'       # improved 8-19 places
        else:                 return 'rising_fast'  # improved 20+ places

    df['volatility_label'] = df['rank_change'].apply(assign_label)

    # Step 8: Remove the first year for each university because cannot calculate
    # lag features for it (there is no previous year to look back at)
    df = df.dropna(subset=['rank_lag1'])

    print(f"\nFinal dataset: {len(df)} rows, {len(df.columns)} columns")
    print("\nVolatility distribution:")
    print(df['volatility_label'].value_counts())
    print("\nUWE Bristol timeline:")
    uwe = df[df['university'] == 'UWE Bristol'][[
        'year','rank','rank_change','volatility_label',
        'satisfaction','satisfaction_change','staff_ratio'
    ]]
    print(uwe.to_string(index=False))
    return df

# Step 9: Run feature engineering on the cleaned data
df_final = engineer_features(df_clean)

In [ ]:
# Step 1: Create the processed data folder if it does not already exist
os.makedirs(SAVE_PATH, exist_ok=True)

# Step 2: Save the full cleaned dataset (all universities, all years)
# This means if Colab disconnects,do not have to re-run the cleaning
full_path = SAVE_PATH + 'cug_clean.csv'
df_final.to_csv(full_path, index=False)
print(f"Saved full dataset: {full_path}")
print(f"Shape: {df_final.shape}")

# Step 3: Save a separate file with only UWE Bristol rows for easy reference
uwe_path = SAVE_PATH + 'uwe_only.csv'
df_final[df_final['university'] == 'UWE Bristol'].to_csv(uwe_path, index=False)
print(f"Saved UWE data: {uwe_path}")

print("\nPipeline complete")

In [ ]:
# Step 1: Define the list of 16 features the model will use to make predictions
# These include raw metrics, lag features, change features, and rolling averages
FEATURES = [
    'entry_tariff',         # how selective the university is (UCAS points)
    'satisfaction',         # student satisfaction score this year
    'staff_ratio',          # students per teacher this year
    'career_prospects',     # graduate job rate this year
    'continuation',         # % students staying for year 2
    'research_quality',     # quality of research output
    'rank_lag1',            # what was the rank last year?
    'satisfaction_lag1',    # what was satisfaction last year?
    'staff_ratio_lag1',     # what was staff ratio last year?
    'career_lag1',          # what were career prospects last year?
    'rank_change',          # how much did rank change this year?
    'satisfaction_change',  # how much did satisfaction change?
    'staff_ratio_change',   # how much did staff ratio change?
    'career_change',        # how much did career prospects change?
    'rank_3yr_avg',         # average rank over the last 3 years (STRONGEST predictor)
    'satisfaction_3yr_avg'  # average satisfaction over the last 3 years
]

# Step 2: Define what are trying to predict — the overall CUG rank
TARGET = 'rank'

# Step 3: Keep only rows where all 16 features and the target are available
df_ml = df_final[FEATURES + [TARGET, 'volatility_label',
                              'university', 'year']].dropna()
print(f"Rows available for ML: {len(df_ml)}")
print(f"Years: {sorted(df_ml['year'].unique())}")

# Step 4: Split the data by TIME — not randomly
# Train = 2016-2024 (the model learns from the past)
# Test  = 2025-2027 (check if it can predict the future)
# A random split would be CHEATING because the model would see future data during training
train_years = [2016,2017,2018,2019,2020,2021,2022,2023,2024]
test_years  = [2025, 2026, 2027]

train = df_ml[df_ml['year'].isin(train_years)]
test  = df_ml[df_ml['year'].isin(test_years)]

print(f"\nTraining rows: {len(train)} ({train_years})")
print(f"Testing rows:  {len(test)} ({test_years})")

# Step 5: Separate features (X) from the target label (y) for both splits
X_train = train[FEATURES]
X_test  = test[FEATURES]
y_train = train[TARGET]
y_test  = test[TARGET]

In [ ]:
# Step 1: Import all the model libraries and metrics need
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import numpy as np
import time

# Step 2: Create an empty dictionary to store each model's results
results = {}

# Step 3: NAIVE BASELINE — the simplest possible prediction
# Just predict that next year's rank will be the same as last year
# If models cannot beat this, they are not useful
naive_preds = X_test['rank_lag1']
naive_mae   = mean_absolute_error(y_test, naive_preds)
results['Naive (last year rank)'] = {
    'MAE' : round(naive_mae, 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, naive_preds)), 2),
    'R2'  : round(r2_score(y_test, naive_preds), 3)
}
print(f"Naive baseline MAE: {naive_mae:.2f} rank positions")

# Step 4: RIDGE REGRESSION — a simple straight-line formula model
# It finds the best weights for: rank = w1*feature1 + w2*feature2 + ...
# alpha=1.0 adds a small penalty to stop the model overfitting
# NOTE: This version includes rank_lag1 which causes it to cheat (see Cell 16)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_preds = ridge.predict(X_test)
ridge_mae   = mean_absolute_error(y_test, ridge_preds)
results['Ridge Regression'] = {
    'MAE' : round(ridge_mae, 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, ridge_preds)), 2),
    'R2'  : round(r2_score(y_test, ridge_preds), 3)
}
print(f"Ridge MAE: {ridge_mae:.2f} rank positions")

# Step 5: RANDOM FOREST — builds 100 decision trees and averages their answers
# Each tree asks a series of yes/no questions about the features
# Averaging 100 trees reduces errors because individual mistakes cancel out
# n_estimators=100  build 100 trees
# max_depth=10 each tree can ask at most 10 yes/no questions
# min_samples_leaf=3 each final answer must cover at least 3 universities
# random_state=42  fixed seed so results are the same every time run
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    min_samples_leaf=3
)

# Step 6: Train the model — it learns which feature combinations predict rank reliably
rf.fit(X_train, y_train)

# Step 7: Use the trained model to predict ranks for the unseen test years
rf_preds = rf.predict(X_test)

# Step 8: Calculate MAE — average number of rank positions the prediction was wrong by
# Example: if actual=[55,67,75] and predicted=[53,65,78]
# errors = [2, 2, 3], MAE = (2+2+3)/3 = 2.33
rf_mae = mean_absolute_error(y_test, rf_preds)
results['Random Forest'] = {
    'MAE' : round(rf_mae, 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, rf_preds)), 2),
    'R2'  : round(r2_score(y_test, rf_preds), 3)
}
print(f"Random Forest MAE: {rf_mae:.2f} rank positions")

# Step 9: XGBOOST — builds trees one after another, each one fixing the previous one's mistakes
# learning_rate=0.1 controls how much each new tree is allowed to correct
# max_depth=6 keeps trees slightly shallower than Random Forest
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_mae   = mean_absolute_error(y_test, xgb_preds)
results['XGBoost'] = {
    'MAE' : round(xgb_mae, 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, xgb_preds)), 2),
    'R2'  : round(r2_score(y_test, xgb_preds), 3)
}
print(f"XGBoost MAE: {xgb_mae:.2f} rank positions")

# Step 10: Print the comparison table so can see all results side by side
print("MODEL COMPARISON RESULTS")
print(f"{'Model':<30} {'MAE':>6} {'RMSE':>7} {'R2':>6}")
for name, scores in results.items():
    print(f"{name:<30} {scores['MAE']:>6} "
          f"{scores['RMSE']:>7} {scores['R2']:>6}")

# Step 11: Draw a bar chart comparing MAE of all four models
# The dashed grey line shows the naive baseline — anything below it means improvement
models = list(results.keys())
maes   = [results[m]['MAE'] for m in models]
plt.figure(figsize=(9, 5))
bars = plt.bar(models, maes, color=['#95a5a6','#3498db','#2ecc71','#e74c3c'])
plt.axhline(y=naive_mae, color='gray', linestyle='--', label='Naive baseline')
plt.ylabel('Mean Absolute Error (rank positions)')
plt.title('Model Comparison: Predicting UK University Rankings\n'
          'Lower MAE = more accurate predictions')
plt.xticks(rotation=15, ha='right')
for bar, mae in zip(bars, maes):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             str(mae), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/model_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("\nNOTE: Ridge R2=1.0 indicates overfitting.")
print("Ridge is memorising training patterns, not generalising.")
print("Random Forest and XGBoost are the reliable models.")

In [ ]:
# Step 1: Import joblib for saving models, os for folders, json for the feature list
import joblib, os, json

# Step 2: Create the models folder on Drive if it does not already exist
save_dir = '/content/drive/MyDrive/uwe_project/models/'
os.makedirs(save_dir, exist_ok=True)

# Step 3: Save the trained LightGBM model to a .pkl file
#         joblib.dump() freezes the model into a file so never need to retrain
joblib.dump(lgbm, save_dir + 'model_lgbm.pkl')
print("Saved LightGBM")

# Step 4: Save the volatility classifier
joblib.dump(clf, save_dir + 'classifier_rf.pkl')
print("Saved classifier")

# Step 5: Save the feature names list so the Flask app knows what order to pass features in
with open(save_dir + 'features.json', 'w') as f:
    json.dump(FEATURES, f)
print("Saved features")

# Step 6: Save the processed dataset so the Flask app can access it
df_ml.to_csv(save_dir + 'df_ml.csv', index=False)
print("Saved data")

print("\nDone.")

In [ ]:
# Step 1: Import SHAP — the library that explains why the model made each prediction
# SHAP assigns each feature a positive or negative contribution value
# Positive = this feature pushed the predicted rank HIGHER (worse)
# Negative = this feature pushed the predicted rank LOWER (better)
import shap
import matplotlib.pyplot as plt
import matplotlib
print("Running SHAP analysis... (takes ~30 seconds)")

# Step 2: Build a SHAP explainer for Random Forest model
# TreeExplainer is the fastest method for tree-based models like Random Forest
explainer   = shap.TreeExplainer(rf)

# Step 3: Calculate SHAP values for every row in the test set
# shap_values[i][j] = how much feature j influenced prediction for university i
shap_values = explainer.shap_values(X_test)

# Step 4: Draw a GLOBAL importance bar chart
# Shows which features matter most on average across all universities
# Taller bar = more influential feature
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Which metrics most strongly predict UK university rank?\n"
          "SHAP Global Feature Importance", fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/shap_global.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_global.png")

# Step 5: Draw a DIRECTION plot
# Each dot is one university-year in the test set
# Red = high feature value, Blue = low feature value
# Left of zero = pushed rank down (better), Right = pushed rank up (worse)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Direction Plot: How each metric affects rank", fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/shap_direction.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_direction.png")

# Step 6: Draw WATERFALL plots specifically for UWE Bristol's test years
# A waterfall shows the chain of reasoning: starting from the average prediction,
# each feature either pushes the rank up (red) or down (blue)
uwe_test_rows = test[test['university'] == 'UWE Bristol']

if len(uwe_test_rows) > 0:
    print(f"\nFound {len(uwe_test_rows)} UWE rows in test set")

    for i, (idx, row) in enumerate(uwe_test_rows.iterrows()):
        pos = list(X_test.index).index(idx)
        actual    = row['rank']
        predicted = rf_preds[pos]
        year_val  = int(row['year'])

        print(f"\nUWE Bristol {year_val}:")
        print(f"  Actual rank:    {int(actual)}")
        print(f"  Predicted rank: {predicted:.1f}")
        print(f"  Error:          {abs(actual - predicted):.1f} places")

        plt.figure(figsize=(11, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values        = shap_values[pos],
                base_values   = explainer.expected_value,
                data          = X_test.iloc[pos],
                feature_names = FEATURES
            ),
            show=False
        )
        plt.title(
            f"UWE Bristol {year_val}: Why did the model predict rank {predicted:.0f}?\n"
            f"(Actual rank: {int(actual)})", fontsize=11
        )
        plt.tight_layout()
        plt.savefig(
            f'/content/drive/MyDrive/uwe_project/shap_uwe_{year_val}.png',
            dpi=150, bbox_inches='tight'
        )
        plt.show()
        print(f"Saved: shap_uwe_{year_val}.png")
else:
    # Step 7: If UWE is not in the test set, run SHAP on the full dataset instead
    print("UWE not in test set — running SHAP on full dataset")
    shap_all = explainer.shap_values(df_ml[FEATURES])
    uwe_all  = df_ml[df_ml['university'] == 'UWE Bristol']
    for i, (idx, row) in enumerate(uwe_all.iterrows()):
        pos      = list(df_ml.index).index(idx)
        year_val = int(row['year'])
        plt.figure(figsize=(11, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values        = shap_all[pos],
                base_values   = explainer.expected_value,
                data          = df_ml[FEATURES].iloc[pos],
                feature_names = FEATURES
            ),
            show=False
        )
        plt.title(f"UWE Bristol {year_val}: SHAP Explanation (rank {int(row['rank'])})")
        plt.tight_layout()
        plt.savefig(
            f'/content/drive/MyDrive/uwe_project/shap_uwe_{year_val}.png',
            dpi=150, bbox_inches='tight'
        )
        plt.show()

print("\nSHAP analysis complete")

In [ ]:
# Step 1: Import the classifier and evaluation tools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Step 2: Copy the ML dataset and create the NEXT YEAR label
# .shift(-1) shifts the volatility_label backwards by one year
# So for a 2023 row, 'next_year_label' = what actually happened in 2024
# This lets the model learn: "given 2023 metrics, predict 2024 outcome"
df_class = df_ml.copy()
df_class['next_year_label'] = (
    df_class.groupby('university')['volatility_label'].shift(-1)
)
df_class = df_class.dropna(subset=['next_year_label'])

# Step 3: Simplify the 5 volatility categories down to just 3
# at_risk   = falling or collapsing (the thing most want to detect)
# improving = rising or rising_fast
# stable    = stable
def simplify_label(label):
    if label in ['falling', 'collapsing']:   return 'at_risk'
    elif label in ['rising', 'rising_fast']: return 'improving'
    else:                                     return 'stable'

df_class['next_year_simple'] = df_class['next_year_label'].apply(simplify_label)
df_class['this_year_simple'] = df_class['volatility_label'].apply(simplify_label)

print("Simplified label distribution:")
print(df_class['next_year_simple'].value_counts())

# Step 4: Temporal split — same principle as the regression model
train_years_c = [2016,2017,2018,2019,2020,2021,2022,2023,2024]
test_years_c  = [2025, 2026]
train_c = df_class[df_class['year'].isin(train_years_c)]
test_c  = df_class[df_class['year'].isin(test_years_c)]
X_train_c = train_c[FEATURES]
X_test_c  = test_c[FEATURES]
y_train_c = train_c['next_year_simple']
y_test_c  = test_c['next_year_simple']
print(f"\nTraining rows: {len(train_c)}")
print(f"Testing rows:  {len(test_c)}")

# Step 5: Train the classifier
# class_weight='balanced' tells the model to pay extra attention to the
# rare 'at_risk' class — without this it would mostly predict 'stable'
# and miss almost all genuine at-risk institutions
clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    max_depth=8,
    min_samples_leaf=2
)
clf.fit(X_train_c, y_train_c)
clf_preds = clf.predict(X_test_c)

# Step 6: Print the classification report showing precision, recall and F1 for each class
# Recall is the most important metric — it tells what % of actual
# at_risk institutions the model successfully detected
print("\nClassification Report (3 categories):")
print(classification_report(y_test_c, clf_preds))

# Step 7: Draw a confusion matrix — rows = actual class, columns = predicted class
# The diagonal shows correct predictions
# Off-diagonal shows what the model got wrong
labels_simple = ['at_risk', 'stable', 'improving']
labels_simple = [l for l in labels_simple if l in y_test_c.values]
cm = confusion_matrix(y_test_c, clf_preds, labels=labels_simple)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=labels_simple, yticklabels=labels_simple, cmap='Blues')
plt.title('Early Warning Classifier: Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/confusion_matrix.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrix.png")

# Step 8: Check how the early warning system would have performed specifically for UWE Bristol
uwe_c = df_class[df_class['university'] == 'UWE Bristol'].copy()
uwe_c['predicted_next'] = clf.predict(uwe_c[FEATURES])
print(uwe_c[['year','rank','this_year_simple','next_year_simple','predicted_next']].to_string(index=False))

In [ ]:
# Step 1: Install imbalanced-learn — the library that contains SMOTE
#         SMOTE stands for Synthetic Minority Over-sampling Technique
!pip install imbalanced-learn -q
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Step 2: Rebuild the classification dataset (self-contained cell)
df_class = df_ml.copy()
df_class['next_year_label'] = (
    df_class.groupby('university')['volatility_label'].shift(-1)
)
df_class = df_class.dropna(subset=['next_year_label'])

def simplify_label(label):
    if label in ['falling', 'collapsing']:   return 'at_risk'
    elif label in ['rising', 'rising_fast']: return 'improving'
    else:                                     return 'stable'

df_class['next_year_simple'] = df_class['next_year_label'].apply(simplify_label)
train_c = df_class[df_class['year'].isin([2016,2017,2018,2019,2020,2021,2022,2023,2024])]
test_c  = df_class[df_class['year'].isin([2025, 2026])]
X_train_c = train_c[FEATURES]
X_test_c  = test_c[FEATURES]
y_train_c = train_c['next_year_simple']
y_test_c  = test_c['next_year_simple']

# Step 3: Apply SMOTE to the TRAINING set only — never to the test set
# SMOTE creates artificial extra examples of the rare 'at_risk' class by:
# 1. Taking a real at_risk university-year row
# 2. Finding its 3 nearest neighbours (k_neighbors=3)
# 3. Creating a new synthetic point somewhere between them
# this gives the model more at_risk examples to learn from
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train_c, y_train_c)

# Step 4: Show the class counts before and after SMOTE
# After SMOTE, all three classes should have roughly equal representation
print("Before SMOTE:")
print(pd.Series(y_train_c).value_counts())
print("\nAfter SMOTE:")
print(pd.Series(y_train_sm).value_counts())

# Step 5: Train a new classifier using the SMOTE-balanced training data
clf_smote = RandomForestClassifier(
    n_estimators=200, random_state=42, max_depth=8, min_samples_leaf=2
)
clf_smote.fit(X_train_sm, y_train_sm)
preds_smote = clf_smote.predict(X_test_c)

print("\nClassification Report after SMOTE:")
print(classification_report(y_test_c, preds_smote))

# Step 6: Check UWE Bristol predictions with the SMOTE model
uwe_c = df_class[df_class['university'] == 'UWE Bristol'].copy()
uwe_c['predicted_smote'] = clf_smote.predict(uwe_c[FEATURES])
print("\nUWE Bristol Early Warning (SMOTE):")
print(uwe_c[['year','rank','next_year_simple','predicted_smote']].to_string(index=False))

In [ ]:
# Step 1: Import precision, recall and F1 scoring tools
from sklearn.metrics import precision_score, recall_score, classification_report

# Step 2: Rebuild the classification data (self-contained cell)
df_class_14 = df_ml.copy()
df_class_14['next_year_label'] = (
    df_class_14.groupby('university')['volatility_label'].shift(-1)
)
df_class_14 = df_class_14.dropna(subset=['next_year_label'])

def simplify_label(label):
    if label in ['falling', 'collapsing']:   return 'at_risk'
    elif label in ['rising', 'rising_fast']: return 'improving'
    else:                                     return 'stable'

df_class_14['next_year_simple'] = df_class_14['next_year_label'].apply(simplify_label)
test_c_14   = df_class_14[df_class_14['year'].isin([2025, 2026])]
X_test_c_14 = test_c_14[FEATURES]
y_test_c_14 = test_c_14['next_year_simple']

# Step 3: Get the raw at_risk PROBABILITY for each test university
# Instead of a hard yes/no prediction, get a probability between 0 and 1
# e.g. 0.3 means "30% chance this institution is at_risk next year"
proba         = clf.predict_proba(X_test_c_14)
classes       = clf.classes_
at_risk_idx   = list(classes).index('at_risk')
at_risk_proba = proba[:, at_risk_idx]

# Step 4: Test different threshold values from 0.15 to 0.50
# Default threshold is 0.50 — only flags at_risk if probability > 50%
# For an early warning system,want to be more cautious
# So test lower thresholds: flag at_risk if probability > 25%, etc.
# RECALL = what % of actual at_risk cases did catch? (most important)
# PRECISION = of the warnings raised, what % were correct?
# F1 = balance between precision and recall
print("Threshold sensitivity for at_risk detection:")
print(f"{'Threshold':<12} {'Precision':>10} {'Recall':>10} {'F1':>8}")

best_threshold = 0.30
best_f1 = 0

for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    preds_t = [
        'at_risk' if p >= t
        else clf.predict(X_test_c_14.iloc[[i]])[0]
        for i, p in enumerate(at_risk_proba)
    ]
    p  = precision_score(y_test_c_14, preds_t,
                         labels=['at_risk'], average='macro', zero_division=0)
    r  = recall_score(y_test_c_14, preds_t,
                      labels=['at_risk'], average='macro', zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"{t:<12} {p:>10.3f} {r:>10.3f} {f1:>8.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

# Step 5: Print the full results at the best threshold
print(f"\nBest threshold by F1: {best_threshold}")
print(f"\nFull classification report at threshold {best_threshold}:")
preds_best = [
    'at_risk' if p >= best_threshold
    else clf.predict(X_test_c_14.iloc[[i]])[0]
    for i, p in enumerate(at_risk_proba)
]
print(classification_report(y_test_c_14, preds_best))

In [ ]:
# Step 1: Import XGBoost classifier and tools to handle text labels
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from collections import Counter

# Step 2: Rebuild classification data (self-contained cell)
df_class_15 = df_ml.copy()
df_class_15['next_year_label'] = (
    df_class_15.groupby('university')['volatility_label'].shift(-1)
)
df_class_15 = df_class_15.dropna(subset=['next_year_label'])

def simplify_label(label):
    if label in ['falling', 'collapsing']:   return 'at_risk'
    elif label in ['rising', 'rising_fast']: return 'improving'
    else:                                     return 'stable'

df_class_15['next_year_simple'] = df_class_15['next_year_label'].apply(simplify_label)
train_c_15 = df_class_15[df_class_15['year'].isin([2016,2017,2018,2019,2020,2021,2022,2023,2024])]
test_c_15  = df_class_15[df_class_15['year'].isin([2025, 2026])]
X_train_c_15 = train_c_15[FEATURES]
X_test_c_15  = test_c_15[FEATURES]
y_train_c_15 = train_c_15['next_year_simple']
y_test_c_15  = test_c_15['next_year_simple']

# Step 3: Convert string labels to numbers because XGBoost needs numeric class labels
# at_risk=0, improving=1, stable=2 (alphabetical order)
# inverse_transform at the end converts back to readable labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_c_15)

# Step 4: Build and train the XGBoost classifier
# scale_pos_weight=3 tells XGBoost to treat at_risk examples as 3x more important
# This is XGBoost's built-in way of handling class imbalance
xgb_clf = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    verbosity=0,
    scale_pos_weight=3
)
xgb_clf.fit(X_train_c_15, y_train_enc)

# Step 5: Make predictions and convert numeric labels back to text
xgb_preds_enc = xgb_clf.predict(X_test_c_15)
xgb_preds = le.inverse_transform(xgb_preds_enc)

print("XGBoost Classifier Results:")
print(classification_report(y_test_c_15, xgb_preds))

# Step 6: Check UWE Bristol predictions with the XGBoost classifier
uwe_15 = df_class_15[df_class_15['university'] == 'UWE Bristol'].copy()
uwe_preds_enc = xgb_clf.predict(uwe_15[FEATURES])
uwe_15['predicted_xgb'] = le.inverse_transform(uwe_preds_enc)
print("\nUWE Bristol Early Warning (XGBoost):")
print(uwe_15[['year','rank','next_year_simple','predicted_xgb']].to_string(index=False))

In [ ]:
# Step 1: Import Ridge and metrics
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Step 2: The original Ridge result in Cell 10 was suspiciously perfect (MAE ≈ 0)
# This happened because rank_lag1 (last year's rank) was in the features
# Since rank barely changes year to year, Ridge just learned to copy rank_lag1
# This is not genuine prediction — it is cheating
# fix it by removing rank_lag1 from Ridge's feature list
FEATURES_NO_LAG = [f for f in FEATURES if f != 'rank_lag1']

# Step 3: Create new train and test sets without rank_lag1
X_train_ridge = train[FEATURES_NO_LAG]
X_test_ridge  = test[FEATURES_NO_LAG]

# Step 4: Retrain Ridge without the leaking feature
ridge_fixed = Ridge(alpha=1.0)
ridge_fixed.fit(X_train_ridge, y_train)
ridge_fixed_preds = ridge_fixed.predict(X_test_ridge)
ridge_fixed_mae = mean_absolute_error(y_test, ridge_fixed_preds)

# Step 5: Print the honest Ridge result and compare with other models
print("Ridge Regression (without rank_lag1):")
print(f"  MAE:  {ridge_fixed_mae:.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, ridge_fixed_preds)):.2f}")
print(f"  R2:   {r2_score(y_test, ridge_fixed_preds):.3f}")
print(f"\nFor comparison:")
print(f"  Naive baseline MAE:     5.99")
print(f"  Ridge (fixed) MAE:      {ridge_fixed_mae:.2f}")
print(f"  Random Forest MAE:      2.05")
print(f"  XGBoost MAE:            1.89")

In [ ]:
# Step 1: Install LightGBM and CatBoost — two more powerful gradient boosting models
!pip install lightgbm catboost -q

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time, numpy as np

extended = {}

# Step 2: LIGHTGBM — uses histogram-based splitting which is faster than XGBoost
# Instead of checking every possible split point, it groups values into bins
# This makes training faster while keeping accuracy the same or better
start = time.time()
lgbm = LGBMRegressor(
    n_estimators=100, learning_rate=0.1,
    max_depth=6, random_state=42, verbose=-1
)
lgbm.fit(X_train, y_train)
lgbm_preds = lgbm.predict(X_test)
lgbm_time = round(time.time() - start, 2)
extended['LightGBM'] = {
    'MAE':  round(mean_absolute_error(y_test, lgbm_preds), 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, lgbm_preds)), 2),
    'R2':   round(r2_score(y_test, lgbm_preds), 3),
    'Time': lgbm_time
}

# Step 3: CATBOOST — gradient boosting designed for categorical features
# Uses ordered boosting to reduce overfitting
# verbose=0 means do not print the training progress
start = time.time()
cat = CatBoostRegressor(
    iterations=100, depth=6, learning_rate=0.1,
    random_state=42, verbose=0
)
cat.fit(X_train, y_train)
cat_preds = cat.predict(X_test)
cat_time = round(time.time() - start, 2)
extended['CatBoost'] = {
    'MAE':  round(mean_absolute_error(y_test, cat_preds), 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, cat_preds)), 2),
    'R2':   round(r2_score(y_test, cat_preds), 3),
    'Time': cat_time
}

# Step 4: NEURAL NETWORK (MLP — Multi-Layer Perceptron)
# Three layers of connected nodes: 64 → 32 → 16 → output
# IMPORTANT: Neural networks are sensitive to feature scale
# A feature with values 0-300 (entry_tariff) would dominate one with values 1-4 (satisfaction)
# So use StandardScaler to rescale all features to mean=0, standard deviation=1
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on training data only
X_test_sc  = scaler.transform(X_test)        # apply same scaling to test data

start = time.time()
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32, 16),  # 3 hidden layers with 64, 32, 16 nodes
    max_iter=2000,                     # maximum number of training passes
    random_state=42,
    early_stopping=True,               # stop training if validation loss stops improving
    validation_fraction=0.1            # use 10% of training data to monitor for early stopping
)
mlp.fit(X_train_sc, y_train)
mlp_preds = mlp.predict(X_test_sc)
mlp_time = round(time.time() - start, 2)
extended['Neural Network (MLP)'] = {
    'MAE':  round(mean_absolute_error(y_test, mlp_preds), 2),
    'RMSE': round(np.sqrt(mean_squared_error(y_test, mlp_preds)), 2),
    'R2':   round(r2_score(y_test, mlp_preds), 3),
    'Time': mlp_time
}

# Step 5: Print the full comparison table including all previous models
print("EXTENDED MODEL COMPARISON")
print(f"{'Model':<28} {'MAE':>6} {'RMSE':>7} {'R2':>7} {'Time(s)':>9}")
prev = {
    'Naive baseline':  {'MAE':5.99, 'RMSE':8.81, 'R2':0.945, 'Time':'-'},
    'Ridge (fixed)':   {'MAE':0.00, 'RMSE':0.00, 'R2':1.000, 'Time':'-'},
    'Random Forest':   {'MAE':2.05, 'RMSE':3.09, 'R2':0.993, 'Time':'-'},
    'XGBoost':         {'MAE':1.89, 'RMSE':2.64, 'R2':0.995, 'Time':'-'},
}
all_models = {**prev, **extended}
for name, scores in all_models.items():
    print(f"{name:<28} {scores['MAE']:>6} "
          f"{scores['RMSE']:>7} {scores['R2']:>7} "
          f"{str(scores['Time']):>9}")

In [ ]:
# Step 1: Import SHAP for explainability analysis best model
import shap
import matplotlib.pyplot as plt
print("Running SHAP on LightGBM...")

# Step 2: Build SHAP explainer for LightGBM
# LightGBM won the model comparison with the lowest MAE (1.76)
# So now explain this best model specifically
lgbm_explainer = shap.TreeExplainer(lgbm)
lgbm_shap      = lgbm_explainer.shap_values(X_test)

# Step 3: Draw the global importance chart for LightGBM
# This shows which features the best model relies on most
plt.figure(figsize=(10, 6))
shap.summary_plot(lgbm_shap, X_test, plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance (LightGBM)\n"
          "Which metrics most strongly predict UK university rank?", fontsize=12)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/shap_lgbm_global.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_lgbm_global.png")

# Step 4: Draw the direction plot for LightGBM
# Shows whether high or low values of each feature push rank up or down
plt.figure(figsize=(10, 8))
shap.summary_plot(lgbm_shap, X_test, show=False)
plt.title("SHAP Direction Plot (LightGBM)", fontsize=12)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/uwe_project/shap_lgbm_direction.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_lgbm_direction.png")

# Step 5: Draw waterfall plots for UWE Bristol specifically using LightGBM
# These are the plots used in the final report
# They show exactly why the model predicted each year's rank for UWE
uwe_test_rows = test[test['university'] == 'UWE Bristol']
print(f"\nUWE Bristol in test set: {len(uwe_test_rows)} rows")

for idx, row in uwe_test_rows.iterrows():
    pos       = list(X_test.index).index(idx)
    year_val  = int(row['year'])
    actual    = int(row['rank'])
    predicted = round(lgbm.predict(X_test.iloc[[pos]])[0], 1)

    print(f"\nUWE {year_val}: actual {actual}, predicted {predicted}")

    # Step 6: Draw the waterfall chart for this specific year
    # The chart starts from the average prediction for all universities
    # Then shows how each feature pushed UWE's prediction up or down
    plt.figure(figsize=(11, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values        = lgbm_shap[pos],
            base_values   = lgbm_explainer.expected_value,
            data          = X_test.iloc[pos],
            feature_names = FEATURES
        ),
        show=False
    )
    plt.title(
        f"UWE Bristol {year_val}: Why predicted rank {predicted}? "
        f"(Actual: {actual})", fontsize=11
    )
    plt.tight_layout()
    plt.savefig(
        f'/content/drive/MyDrive/uwe_project/shap_lgbm_uwe_{year_val}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    print(f"Saved: shap_lgbm_uwe_{year_val}.png")

print("\nSHAP on LightGBM complete")